### Цели и задачи проекта

Цели проекта:
Использовать на практике умение предобработки данных: очистка, типизация, фильтрация и категоризация, используя Pandas в среде разработки Jupyter Notebook.
Подготовить данные о продажах игр за 2000–2013г. для анализа необходимого для статьи.

Задачи проекта:
Визуализация и анализ сырых данных
Предобработка данных
Разбиение по сегментам, фильтрация

### Описание данных
Исходные данные таблицы включают:
Platform — название платформы.
Year of Release — год выпуска игры.
Genre — жанр игры.
NA sales — продажи в Северной Америке (в миллионах проданных копий).
EU sales — продажи в Европе (в миллионах проданных копий).
JP sales — продажи в Японии (в миллионах проданных копий).
Other sales — продажи в других странах (в миллионах проданных копий).
Critic Score — оценка критиков (от 0 до 100).
User Score — оценка пользователей (от 0 до 10).
Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

<font color='#777778'>Перечислите основные шаги проекта или напишите оглавление. Используйте описание проекта, чтобы зафиксировать основные шаги.</font>

---

## 1. Загрузка данных и знакомство с ними

- Загрузите необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [50]:
import pandas as pd
df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

- Познакомьтесь с данными: выведите первые строки и результат метода `info()`.


In [51]:
print(df.head())
initial_rows_count = len(df)

                       Name Platform  Year of Release         Genre  NA sales  \
0                Wii Sports      Wii           2006.0        Sports     41.36   
1         Super Mario Bros.      NES           1985.0      Platform     29.08   
2            Mario Kart Wii      Wii           2008.0        Racing     15.68   
3         Wii Sports Resort      Wii           2009.0        Sports     15.61   
4  Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing     11.27   

  EU sales JP sales  Other sales  Critic Score User Score Rating  
0    28.96     3.77         8.45          76.0          8      E  
1     3.58     6.81         0.77           NaN        NaN    NaN  
2    12.76     3.79         3.29          82.0        8.3      E  
3    10.93     3.28         2.95          80.0          8      E  
4     8.89    10.22         1.00           NaN        NaN    NaN  


In [52]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB
None


##### Вывод по предоставленным данным:
- Все числовые данные, включая год, содержатся в типе float
- Данные об игре могут не содержать информацию об оценке, более половины не имеет оценку от критиков
- Есть 2 игры без названия и жанра, но с данными о продажах что странно
- Датафрейм имеет объём 1.4 МВ и 16956 строчек 
- User Score, EU, JP sales имеют тип объекта хотя есть необходимоть работы с ними, как с float
- 275 записей не имеют данных о годе выпуска

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

- Выведите на экран названия всех столбцов датафрейма и проверьте их стиль написания.
- Приведите все столбцы к стилю snake case. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [53]:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')


In [54]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.lower().str.replace(' ', '_')

- df.columns = df.columns.str.lower() - переводит все символы нижний регистр
- df.columns = df.columns.str.lower().str.replace(' ', '_') заменяет все символы пробела на подчеркивание

In [55]:
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')


### 2.2. Типы данных

- Если встречаются некорректные типы данных, предположите их причины.
- При необходимости проведите преобразование типов данных. Помните, что столбцы с числовыми данными и пропусками нельзя преобразовать к типу `int64`. Сначала вам понадобится обработать пропуски, а затем преобразовать типы данных.

In [56]:
mask_user = pd.to_numeric(df['user_score'], errors='coerce').isna() & df['user_score'].notna()
non_numeric_user = df.loc[mask_user, 'user_score']
print('user_score:', set(non_numeric_user), len(non_numeric_user))

mask_eu = pd.to_numeric(df['eu_sales'], errors='coerce').isna() & df['eu_sales'].notna()
non_numeric_eu = df.loc[mask_eu, 'eu_sales']
print('eu_sales:', set(non_numeric_eu), len(non_numeric_eu))

mask_na = pd.to_numeric(df['na_sales'], errors='coerce').isna() & df['na_sales'].notna()
non_numeric_na = df.loc[mask_na, 'na_sales']
print('na_sales:', set(non_numeric_na), len(non_numeric_na))

user_score: {'tbd'} 2464
eu_sales: {'unknown'} 6
na_sales: set() 0


Используя маску pd.to_numeric(df['name'], errors='coerce').isna() & df['name']].notna мы найдем все значения, которые не получилось преобразовать в число, но они есть в столбце

По полученным данным можно определить что: 
- user_score имеет 2464 значений TBD(to be detemined)
- eu_sales имеет значения unknown вместо пропусков 

In [57]:
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
df['eu_sales'] = pd.to_numeric(df['eu_sales'],   errors='coerce')
df['na_sales'] = pd.to_numeric(df['na_sales'],   errors='coerce')

преобразум необходимые значения в числа

### 2.3. Наличие пропусков в данных

- Посчитайте количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [58]:
total = len(df)
for col in df.columns:
    missing = df[col].isnull().sum()
    missing_percent = (missing / total) * 100
    print(col, missing, missing_percent)

name 2 0.01179523472517103
platform 0 0.0
year_of_release 275 1.6218447747110165
genre 2 0.01179523472517103
na_sales 0 0.0
eu_sales 6 0.035385704175513094
jp_sales 0 0.0
other_sales 0 0.0
critic_score 8714 51.39183769757019
user_score 9268 54.659117716442566
rating 6871 40.52252889832508


- Изучите данные с пропущенными значениями. Напишите промежуточный вывод: для каких столбцов характерны пропуски и сколько их. Предположите, почему пропуски могли возникнуть. Укажите, какие действия с этими данными можно сделать и почему.


###### Пропуски характерны для столбцов с
- названием игры
- жанрам
- годом релиза
- рейтингом
- оценками пользователей
- оценками критиков

###### Пропуски могли воникнуть из-за:
- в название и жанре технический сбой
- в годе из-за недостатка данных
- рейтинг может быть не указан
- оценки может не быть из-за малоизвестности игры или по причине отсутствия данных

Действия с пропусками:
- строки c нехваткой года названия и жанра можно удалитьь (< 2%)
- строки с нехваткой рейтинга игнорировать, так как они не используются
- Пропуски в оценке изменить на 'нет оценки' так как их очень много

In [59]:
df = df.dropna(subset=['name', 'year_of_release', 'genre'])

In [60]:
df['user_score'] = df['user_score'].fillna(-1)
df['critic_score'] = df['critic_score'].fillna(-1)
df['year_of_release'] = df['year_of_release'].astype(int)

### 2.4. Явные и неявные дубликаты в данных

- Изучите уникальные значения в категориальных данных, например с названиями жанра игры, платформы, рейтинга и года выпуска. Проверьте, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания.
- При необходимости проведите нормализацию данных с текстовыми значениями. Названия или жанры игр можно привести к нижнему регистру, а названия рейтинга — к верхнему.

In [61]:
print('Жанры: ', df['genre'].unique())
print('Платформы: ',df['platform'].unique())
print('Рейтинг: ',df['rating'].unique())
print('Год: ',df['year_of_release'].unique())
print('Количество дубликатов до обработки неявных: ',sum(df.duplicated()))

Жанры:  <StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy',
         'MISC', 'ROLE-PLAYING',       'RACING',       'ACTION',
      'SHOOTER',     'FIGHTING',       'SPORTS',     'PLATFORM',
    'ADVENTURE',   'SIMULATION',       'PUZZLE',     'STRATEGY']
Length: 24, dtype: str
Платформы:  <StringArray>
[ 'Wii',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  'PS4',  '3DS',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XOne',
 'WiiU',   'GC',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str
Рейтинг:  <StringArray>
['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str
Год:  [2006 1985 2008 2009 1996 1989 1984 2005 1999 2007 2010 2013 2004 1990
 1988 2002 2001 2011 1998 2015 2012 2014 1992 1997 1993 1994 1982 2016
 2003 1986 2000 

Могут быть неявные дубликаты из-за написания жанра в разных регистрах. Изменим регистр и далее все дубликаты станут явными.

In [62]:
df['genre'] = df['genre'].str.lower().str.strip()
print(df['genre'].unique())

<StringArray>
[      'sports',     'platform',       'racing', 'role-playing',
       'puzzle',         'misc',      'shooter',   'simulation',
       'action',     'fighting',    'adventure',     'strategy']
Length: 12, dtype: str


- После того как нормализуете данные и устраните неявные дубликаты, проверьте наличие явных дубликатов в данных.

In [63]:
print('Количество дубликатов: ',sum(df.duplicated()))
df = df.drop_duplicates()

Количество дубликатов:  235


- Напишите промежуточный вывод: укажите количество найденных дубликатов и действия по их обработке.

Всего было найдено 235 дубликатов, из инх 56 возникли из-за записи в разныых регистрах жанра

- В процессе подготовки данных вы могли что-либо удалять, например строки с пропусками или ошибками, дубликаты и прочее. В этом случае посчитайте количество удалённых строк в абсолютном и относительном значениях.

In [64]:
final_rows_count = len(df)
removed_rows = initial_rows_count - final_rows_count  

print('Начальное количество строк:', initial_rows_count, 'После удаления:', final_rows_count)
print('Количество удаленных строк:', removed_rows, 'В процентах:', round(removed_rows / initial_rows_count * 100, 2))


Начальное количество строк: 16956 После удаления: 16444
Количество удаленных строк: 512 В процентах: 3.02


##### Промежуточный вывод
- Данные приведены к необходимому для обработки типу. 
- Решены проблемы пропусков и дубликатов. 
- Пропуски в оценках заменены на 'нет оценки'
- Пропуски в рейтинге оставлены так как не помешают дальнейшему иследованию

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберите данные по этому показателю. Сохраните новый срез данных в отдельном датафрейме, например `df_actual`.

In [65]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()
print(sorted(df_actual['year_of_release'].unique()))

[np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013)]


---

## 4. Категоризация данных
    
Проведите категоризацию данных:
- Разделите все игры по оценкам пользователей и выделите такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [66]:
df['user_score_cat'] = pd.cut(df['user_score'], bins=[-1, 0, 3, 8, 10.001], labels = ['нет оценки', 'низкая', 'средняя', 'высокая'], right=False)

In [67]:
df['critic_score_cat'] = pd.cut(df['critic_score'], bins=[-1, 0, 30, 80, 100.1], labels = ['нет оценки', 'низкая', 'средняя', 'высокая'], right=False)

- После категоризации данных проверьте результат: сгруппируйте данные по выделенным категориям и посчитайте количество игр в каждой категории.

In [68]:
print(df.groupby('user_score_cat', observed=True)['user_score_cat'].count())
print(df.groupby('critic_score_cat', observed=True)['critic_score_cat'].count())

user_score_cat
нет оценки    8981
низкая         142
средняя       4777
высокая       2544
Name: user_score_cat, dtype: int64
critic_score_cat
нет оценки    8461
низкая          59
средняя       5945
высокая       1979
Name: critic_score_cat, dtype: int64


- Выделите топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [69]:
platform_top = df_actual.groupby('platform')['platform'].count().sort_values(ascending=False).head(7)
print(platform_top)

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: platform, dtype: int64


---

## 5. Итоговый вывод

В конце напишите основной вывод и отразите, какую работу проделали. Не забудьте указать описание среза данных и новых полей, которые добавили в исходный датасет.

В проекте была сделанна предобработка и анализ. Датасет имеет столбцы о названиях игр, платформах, годах выпуска, жанрах, продажах в разных регионах, оценках критиков и пользователей и возрастных рейтингах.

Предобработка данных включала в себя:
- Стандартизацию имён столбцов: приведение к стилю snake_case(нижний регистр, подчеркивание всесто пробелов).
- Изменение типов данных: изменены нечисловые значения(user_score, eu_sales, na_sales). Это позволило корректно работать с числовыми данными.
- Обработку пропусков: name, year_of_release, genre cтроки с пропусками были удалены, так как отсутствие года делает записи непригодными для анализа.
user_score и critic_score пропуски > 50%, поэтому они были заполнены строкой 'нет оценки', чтобы сохранить записи, но обозначить отсутствие оценки.
rating оставлен без изменений, так как этот признак не является ключевым для текущего исследования.
- Устранение дубликатов: присутствовали неявные дубликаты в жанрах из-за разного регистра которые были удалены, после приведения жанров к нижнему регистру и удаления явных дубликатов (235 записей) с помощью drop_duplicates().

- Создание новых категориальных полей: Используя user_score создана категория user_score_cat: низкая, средняя 3–8 и высокая 8–10. Аналогично по critic_score создана категория critic_score_cat: низкая, средняя, высокая. Был применён pd.cut.

Фильтрация данных для анализа
- был выделен период с 2000 по 2013 год включительно.

Анализ топ-7 платформ
- Для периода 2000–2013 годов найден топ-7 платформ по количеству выпущенных игр.

Новые поля в датасете
- user_score_cat – категория пользовательской оценки.
- critic_score_cat – категория оценки критиков.